In [1]:
print("Hello World")

Hello World


In [17]:
import numpy as np
import json
import pandas as pd
from utils import data_completeness, data_uniqueness, classification_metrics, validity_check, chi_test, ks_test, determine_dtype_ft

In [37]:
def get_results_dict(model_name, date, baseline_data, gt, production_data): # pd , baselinedata,gt,
    
    """
    Returns a result dictionary that calls all the 5 analyses, puts the results in a dictionary along with some other
    related details (e.g: column type, statistical information)
    """
    
    dc = data_completeness(production_data[gt.drop('target',axis=1).columns])
    du_score, uniq_score, _, len_of_df, len_of_df_wo_dup = data_uniqueness(production_data[gt.drop('target',axis=1).columns])
    cat_has_int, miss_dict, dv_score, indices, outliers_index, _ = validity_check(baseline_data.drop('target',axis=1),production_data.drop('target',axis=1))
    def data_unique(score, uniq_score, len_of_df, len_of_df_wo_dup):
        results = dict()
        results['score'] = round(score,2)
        results['uniquesness_score'] = {uniq_score['Feature'][i]:uniq_score['Value'][i] for i in range(len(uniq_score['Feature']))}
        results["len_of_df"] = len_of_df
        results["len_of_df_without_duplicates"] = len_of_df_wo_dup
        return results
    def data_validity_results(cat_int, miss_dict, score, indices, out_index,df):
        results = dict()
        results['score'] = score
        for i in df.columns :
            results[i] = dict()
            results[i]['missing_values'] = len(miss_dict[i])
            if df[i].dtype == 'object':
                if cat_int[i] != 0:
                    results[i]['datatype_mismatch'] = len(cat_int[i])
                else :
                    results[i]['datatype_mismatch'] = 0
            else : 
                results[i]['datatype_mismatch'] = 0
            results[i]['outliers'] = len(out_index[i])
        results['total_invalid_rows'] = len(set(indices))
        return results
    categorical_ft, numerical_ft = determine_dtype_ft(baseline_data.drop('target',axis=1)) 
    ct,_ = chi_test(baseline_data.drop('target',axis=1), production_data.drop('target',axis=1), categorical_ft, alpha=0.05)
    kt,_ = ks_test(baseline_data.drop('target',axis=1), production_data.drop('target',axis=1), num_ft= numerical_ft, nlp=False, alpha=0.05, b_name="Baseline Data")
    def chi_ks_results(ct, kt, df):
        results = dict()
        for i in ct:
            results[i] = dict()
            results[i]['type'] = 'categorical'
            results[i]['chi'] = ct[i]['chi']
            results[i]['p-value'] = ct[i]['p-value']
            results[i]['status'] = ct[i]['status']
            results[i]['data_statistics'] = dict()
            results[i]['data_statistics']['mode'] = df[i].mode().values.tolist()
            results[i]['data_statistics']['value_counts'] = df[i].value_counts().to_dict()
        for i in kt:
            results[i] = dict()
            results[i]['type'] = "numerical"
            results[i]['ks-stat'] = kt[i]['ks-stat']
            results[i]['p-value'] = kt[i]['p-value']
            results[i]['p-value'] = kt[i]['status']
            q1 = np.nanpercentile(df[i],25)
            q3 = np.nanpercentile(df[i],75)
            iqr_values = q3 - q1
            lower_bound = q1 - 1.5 * iqr_values
            upper_bound = q3 + 1.5 * iqr_values
            outliers = df[i][(df[i] < lower_bound) | (df[i] > upper_bound)]
            results[i]['data_statistics'] = dict()
            results[i]['data_statistics']['mean'] = round(np.nanmean(df[i]),2)
            results[i]['data_statistics']['median'] = round(np.nanmedian(df[i]),2)
            results[i]['data_statistics']['std'] = round(np.nanstd(df[i]),2)
            results[i]['data_statistics']['min'] = round(np.nanmin(df[i]),2)
            results[i]['data_statistics']['max'] = round(np.nanmax(df[i]),2)
            results[i]['data_statistics']['iqr'] = iqr_values
            results[i]['data_statistics']['outlier_count'] = len(outliers)
        return results
    du = data_unique(du_score, uniq_score, len_of_df, len_of_df_wo_dup)
    validity = data_validity_results(cat_has_int, miss_dict, dv_score, indices, outliers_index, production_data[gt.drop('target',axis=1).columns])
    metrics = classification_metrics(gt['target'],production_data['target'])
    with open("../pages/benchmark.json", "r") as bench :
        benchmark_metrics = json.load(bench)
    #print(benchmark_metrics)
    pred_drift_results,_ = chi_test(gt,production_data, categorical_ft=["target"], alpha = 0.05)
    data_drift_results = chi_ks_results(ct, kt, production_data.drop('target',axis=1))
    final_results = { # add column type , full data drfit analysis
    model_name : {
        date : {
            "Data Quality" : {"Data Completeness":dc,"Data Uniqueness":du, "Data Validity":validity },
            "Performance Drift" : {
                "Metrics" : metrics ,  # check fpr 
                "Drifting Metrics" : [i for i in list(benchmark_metrics[model_name].keys()) if metrics[i] < benchmark_metrics[model_name][i]]
            } , 
            "Prediction Drift" : pred_drift_results ,
            
            "Data Drift" :  data_drift_results 
                
            
        }
    }
}
    
    return final_results

In [38]:
production_data = pd.read_csv("./Heart Disease Prediction/Production Runs/31st Jan 2024.csv")

In [39]:
gt = pd.read_csv("./Heart Disease Prediction/Ground Truths/31st Jan 2024.csv")

In [40]:
baseline = pd.read_csv("./Heart Disease Prediction/baseline.csv")

In [41]:
results = get_results_dict("Heart Disease Prediction", "31st Jan 2024", baseline_data=baseline, gt=gt, production_data=production_data)

In [42]:
results

{'Heart Disease Prediction': {'31st Jan 2024': {'Data Quality': {'Data Completeness': (100.0,
     209,
     209,
     0,
     Age               0
     Sex               0
     ChestPainType     0
     RestingBP         0
     Cholesterol       0
     FastingBS         0
     RestingECG        0
     MaxHR             0
     ExerciseAngina    0
     Oldpeak           0
     ST_Slope          0
     dtype: int64),
    'Data Uniqueness': {'score': 100.0,
     'uniquesness_score': {'Age': 20.5742,
      'Sex': 0.9569,
      'ChestPainType': 1.9139,
      'RestingBP': 20.0957,
      'Cholesterol': 53.11,
      'FastingBS': 0.9569,
      'RestingECG': 1.4354,
      'MaxHR': 37.3206,
      'ExerciseAngina': 0.9569,
      'Oldpeak': 14.3541,
      'ST_Slope': 1.4354},
     'len_of_df': 209,
     'len_of_df_without_duplicates': 209},
    'Data Validity': {'score': 100.0,
     'Age': {'missing_values': 0, 'datatype_mismatch': 0, 'outliers': 0},
     'Sex': {'missing_values': 0, 'datatype_mismat

In [44]:
import ollama
import os

In [45]:
os.system("ollama list")

0

In [ ]:
response = ollama.chat(model='heart', messages=[
  {
    'role': 'user',
    'content': 'What is Heparin? Explain it in 30 simple words',
  },
])
print(response['message']['content'])

In [58]:
domain = {'task': 'Predict heart disease presence based on patient characteristics and test results. Data is collected from clinical assessments, medical tests, and patient history.', 'column_details': {'Age': {'data_type': 'int64', 'meaning': 'Age of the patient in years, recorded at the time of assessment.', 'statistics': {'mean': 54.1, 'min': 29, 'max': 77}}, 'Sex': {'data_type': 'object', 'meaning': 'Gender of the patient (M = Male, F = Female), recorded from patient demographics.', 'unique_values': {'M': 381, 'F': 119}}, 'ChestPainType': {'data_type': 'object', 'meaning': 'Type of chest pain experienced, reported by the patient (ASY = Asymptomatic, NAP = Non-anginal pain, ATA = Atypical angina, TA = Typical angina).', 'unique_values': {'ASY': 272, 'NAP': 118, 'ATA': 86, 'TA': 24}}, 'RestingBP': {'data_type': 'int64', 'meaning': 'Resting blood pressure in mmHg, measured during a clinical visit.', 'statistics': {'mean': 133.5, 'min': 0, 'max': 200}}, 'Cholesterol': {'data_type': 'int64', 'meaning': 'Serum cholesterol level in mg/dL, obtained from a blood test.', 'statistics': {'mean': 202.4, 'min': 0, 'max': 603}}, 'FastingBS': {'data_type': 'int64', 'meaning': 'Fasting blood sugar level (1 = true, 0 = false), measured after an overnight fast.', 'statistics': {'mean': 0.226, 'min': 0, 'max': 1}}, 'RestingECG': {'data_type': 'object', 'meaning': 'Results of resting electrocardiogram (Normal, LVH = Left Ventricular Hypertrophy, ST = ST-T wave abnormality), recorded from an ECG test.', 'unique_values': {'Normal': 291, 'LVH': 109, 'ST': 100}}, 'MaxHR': {'data_type': 'int64', 'meaning': 'Maximum heart rate achieved during an exercise stress test.', 'statistics': {'mean': 135.5, 'min': 63, 'max': 192}}, 'ExerciseAngina': {'data_type': 'object', 'meaning': 'Exercise-induced angina (Y = Yes, N = No), determined from stress test observations.', 'unique_values': {'N': 285, 'Y': 215}}, 'Oldpeak': {'data_type': 'float64', 'meaning': 'ST depression induced by exercise relative to rest, measured in an exercise stress test.', 'statistics': {'mean': 0.9954, 'min': -0.8, 'max': 6.2}}, 'ST_Slope': {'data_type': 'object', 'meaning': 'Slope of the peak exercise ST segment (Flat, Up, Down), indicating heart stress response.', 'unique_values': {'Flat': 263, 'Up': 201, 'Down': 36}}, 'target': {'data_type': 'int64', 'meaning': 'Heart disease diagnosis (1 = presence, 0 = absence), determined based on medical examination and test results.', 'statistics': {'mean': 0.542, 'min': 0, 'max': 1}}}}

In [64]:
def create_ollama_model(model_name, model_type, domain_knowledge):
    
    llm_name = "_".join(model_name.lower().split())
    
    template = """{{- if .Messages }}
{{- range $index, $_ := .Messages }}
{{- if eq .Role "user" }}
{{- if and (eq (len (slice $.Messages $index)) 1) $.Tools }}[AVAILABLE_TOOLS] {{ $.Tools }}[/AVAILABLE_TOOLS]
{{- end }}[INST] {{ if and $.System (eq (len (slice $.Messages $index)) 1) }}{{ $.System }}

{{ end }}{{ .Content }}[/INST]
{{- else if eq .Role "assistant" }}
{{- if .Content }} {{ .Content }}
{{- else if .ToolCalls }}[TOOL_CALLS] [
{{- range .ToolCalls }}{"name": "{{ .Function.Name }}", "arguments": {{ .Function.Arguments }}}
{{- end }}]
{{- end }}</s>
{{- else if eq .Role "tool" }}[TOOL_RESULTS] {"content": {{ .Content }}} [/TOOL_RESULTS]
{{- end }}
{{- end }}
{{- else }}[INST] {{ if .System }}{{ .System }}

{{ end }}{{ .Prompt }}[/INST]
{{- end }} {{ .Response }}
{{- if .Response }}</s>
{{- end }}"""
    
    system = f"""You are a very smart, knowledgeable, and helpful assistant that answers questions related to model degradation issues, metrics, and data of a {model_name} application, which is a {model_type} task."

 You are part of a root cause analysis application where machine learning models are deployed and assessed against 5 analyses types:
1) Performance Drift Analysis
2) Prediction Drift Analysis
3) Data Drift Analysis
4) Data Quality Analysis
5) Model Explanations/Interpretations

You'll be provided domain knowledge about the model, its baseline data, and production data (based on the production date).

Domain Knowledge:
{domain_knowledge}
    
Instructions:
1) DO NOT provide false information. Answer only based on available data.
2) Provide ONLY accurate information about metrics or clinical jargon.
3) Keep answers concise (≤30 words).
4) Always be polite, ethical, and safe. NO harmful, illegal, or offensive responses.
5) If you don’t know the answer, admit it instead of guessing.
6) Suggest potential root causes for model degradation while following all other guidelines.

Your name is {llm_name}_mistralLLM.

You'll be given information about the analysis results for the production run in the prompt."""
    
    
    license = """                                 Apache License
                           Version 2.0, January 2004
                        http://www.apache.org/licenses/

   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION

   1. Definitions.

      "License" shall mean the terms and conditions for use, reproduction,
      and distribution as defined by Sections 1 through 9 of this document.

      "Licensor" shall mean the copyright owner or entity authorized by
      the copyright owner that is granting the License.

      "Legal Entity" shall mean the union of the acting entity and all
      other entities that control, are controlled by, or are under common
      control with that entity. For the purposes of this definition,
      "control" means (i) the power, direct or indirect, to cause the
      direction or management of such entity, whether by contract or
      otherwise, or (ii) ownership of fifty percent (50%) or more of the
      outstanding shares, or (iii) beneficial ownership of such entity.

      "You" (or "Your") shall mean an individual or Legal Entity
      exercising permissions granted by this License.

      "Source" form shall mean the preferred form for making modifications,
      including but not limited to software source code, documentation
      source, and configuration files.

      "Object" form shall mean any form resulting from mechanical
      transformation or translation of a Source form, including but
      not limited to compiled object code, generated documentation,
      and conversions to other media types.

      "Work" shall mean the work of authorship, whether in Source or
      Object form, made available under the License, as indicated by a
      copyright notice that is included in or attached to the work
      (an example is provided in the Appendix below).

      "Derivative Works" shall mean any work, whether in Source or Object
      form, that is based on (or derived from) the Work and for which the
      editorial revisions, annotations, elaborations, or other modifications
      represent, as a whole, an original work of authorship. For the purposes
      of this License, Derivative Works shall not include works that remain
      separable from, or merely link (or bind by name) to the interfaces of,
      the Work and Derivative Works thereof.

      "Contribution" shall mean any work of authorship, including
      the original version of the Work and any modifications or additions
      to that Work or Derivative Works thereof, that is intentionally
      submitted to Licensor for inclusion in the Work by the copyright owner
      or by an individual or Legal Entity authorized to submit on behalf of
      the copyright owner. For the purposes of this definition, "submitted"
      means any form of electronic, verbal, or written communication sent
      to the Licensor or its representatives, including but not limited to
      communication on electronic mailing lists, source code control systems,
      and issue tracking systems that are managed by, or on behalf of, the
      Licensor for the purpose of discussing and improving the Work, but
      excluding communication that is conspicuously marked or otherwise
      designated in writing by the copyright owner as "Not a Contribution."

      "Contributor" shall mean Licensor and any individual or Legal Entity
      on behalf of whom a Contribution has been received by Licensor and
      subsequently incorporated within the Work.

   2. Grant of Copyright License. Subject to the terms and conditions of
      this License, each Contributor hereby grants to You a perpetual,
      worldwide, non-exclusive, no-charge, royalty-free, irrevocable
      copyright license to reproduce, prepare Derivative Works of,
      publicly display, publicly perform, sublicense, and distribute the
      Work and such Derivative Works in Source or Object form.

   3. Grant of Patent License. Subject to the terms and conditions of
      this License, each Contributor hereby grants to You a perpetual,
      worldwide, non-exclusive, no-charge, royalty-free, irrevocable
      (except as stated in this section) patent license to make, have made,
      use, offer to sell, sell, import, and otherwise transfer the Work,
      where such license applies only to those patent claims licensable
      by such Contributor that are necessarily infringed by their
      Contribution(s) alone or by combination of their Contribution(s)
      with the Work to which such Contribution(s) was submitted. If You
      institute patent litigation against any entity (including a
      cross-claim or counterclaim in a lawsuit) alleging that the Work
      or a Contribution incorporated within the Work constitutes direct
      or contributory patent infringement, then any patent licenses
      granted to You under this License for that Work shall terminate
      as of the date such litigation is filed.

   4. Redistribution. You may reproduce and distribute copies of the
      Work or Derivative Works thereof in any medium, with or without
      modifications, and in Source or Object form, provided that You
      meet the following conditions:

      (a) You must give any other recipients of the Work or
          Derivative Works a copy of this License; and

      (b) You must cause any modified files to carry prominent notices
          stating that You changed the files; and

      (c) You must retain, in the Source form of any Derivative Works
          that You distribute, all copyright, patent, trademark, and
          attribution notices from the Source form of the Work,
          excluding those notices that do not pertain to any part of
          the Derivative Works; and

      (d) If the Work includes a "NOTICE" text file as part of its
          distribution, then any Derivative Works that You distribute must
          include a readable copy of the attribution notices contained
          within such NOTICE file, excluding those notices that do not
          pertain to any part of the Derivative Works, in at least one
          of the following places: within a NOTICE text file distributed
          as part of the Derivative Works; within the Source form or
          documentation, if provided along with the Derivative Works; or,
          within a display generated by the Derivative Works, if and
          wherever such third-party notices normally appear. The contents
          of the NOTICE file are for informational purposes only and
          do not modify the License. You may add Your own attribution
          notices within Derivative Works that You distribute, alongside
          or as an addendum to the NOTICE text from the Work, provided
          that such additional attribution notices cannot be construed
          as modifying the License.

      You may add Your own copyright statement to Your modifications and
      may provide additional or different license terms and conditions
      for use, reproduction, or distribution of Your modifications, or
      for any such Derivative Works as a whole, provided Your use,
      reproduction, and distribution of the Work otherwise complies with
      the conditions stated in this License.

   5. Submission of Contributions. Unless You explicitly state otherwise,
      any Contribution intentionally submitted for inclusion in the Work
      by You to the Licensor shall be under the terms and conditions of
      this License, without any additional terms or conditions.
      Notwithstanding the above, nothing herein shall supersede or modify
      the terms of any separate license agreement you may have executed
      with Licensor regarding such Contributions.

   6. Trademarks. This License does not grant permission to use the trade
      names, trademarks, service marks, or product names of the Licensor,
      except as required for reasonable and customary use in describing the
      origin of the Work and reproducing the content of the NOTICE file.

   7. Disclaimer of Warranty. Unless required by applicable law or
      agreed to in writing, Licensor provides the Work (and each
      Contributor provides its Contributions) on an "AS IS" BASIS,
      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or
      implied, including, without limitation, any warranties or conditions
      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A
      PARTICULAR PURPOSE. You are solely responsible for determining the
      appropriateness of using or redistributing the Work and assume any
      risks associated with Your exercise of permissions under this License.

   8. Limitation of Liability. In no event and under no legal theory,
      whether in tort (including negligence), contract, or otherwise,
      unless required by applicable law (such as deliberate and grossly
      negligent acts) or agreed to in writing, shall any Contributor be
      liable to You for damages, including any direct, indirect, special,
      incidental, or consequential damages of any character arising as a
      result of this License or out of the use or inability to use the
      Work (including but not limited to damages for loss of goodwill,
      work stoppage, computer failure or malfunction, or any and all
      other commercial damages or losses), even if such Contributor
      has been advised of the possibility of such damages.

   9. Accepting Warranty or Additional Liability. While redistributing
      the Work or Derivative Works thereof, You may choose to offer,
      and charge a fee for, acceptance of support, warranty, indemnity,
      or other liability obligations and/or rights consistent with this
      License. However, in accepting such obligations, You may act only
      on Your own behalf and on Your sole responsibility, not on behalf
      of any other Contributor, and only if You agree to indemnify,
      defend, and hold each Contributor harmless for any liability
      incurred by, or claims asserted against, such Contributor by reason
      of your accepting any such warranty or additional liability.

   END OF TERMS AND CONDITIONS

   APPENDIX: How to apply the Apache License to your work.

      To apply the Apache License to your work, attach the following
      boilerplate notice, with the fields enclosed by brackets "[]"
      replaced with your own identifying information. (Don't include
      the brackets!)  The text should be enclosed in the appropriate
      comment syntax for the file format. We also recommend that a
      file or class name and description of purpose be included on the
      same "printed page" as the copyright notice for easier
      identification within third-party archives.

   Copyright [yyyy] [name of copyright owner]

   Licensed under the Apache License, Version 2.0 (the "License");
   you may not use this file except in compliance with the License.
   You may obtain a copy of the License at

       http://www.apache.org/licenses/LICENSE-2.0

   Unless required by applicable law or agreed to in writing, software
   distributed under the License is distributed on an "AS IS" BASIS,
   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
   See the License for the specific language governing permissions and
   limitations under the License."""


    
    
    modelfile = f'''
    # Modelfile generated by "ollama show"
# To build a new Modelfile based on this, replace FROM with:
# FROM mistral:latest

FROM C:\\Users\\Akshat Mittu\\.ollama\\models\\blobs\\sha256-ff82381e2bea77d91c1b824c7afb83f6fb73e9f7de9dda631bcdbca564aa5435
TEMPLATE """{template}"""

PARAMETER stop [INST]
PARAMETER stop [/INST]

SYSTEM """{system}"""
LICENSE """{license}"""
'''
    with open(f"{llm_name}.modelfile", "w", encoding="utf-8") as m:
        m.write(modelfile)
    
    os.system(f"ollama create {llm_name}_mistralLLM --file {llm_name}.modelfile")
    
    
    return "Success"

In [62]:
print(create_ollama_model("Heart Disease Prediction", "Classification", domain))


    # Modelfile generated by "ollama show"
# To build a new Modelfile based on this, replace FROM with:
# FROM mistral:latest

FROM C:\Users\Akshat Mittu\.ollama\models\blobs\sha256-ff82381e2bea77d91c1b824c7afb83f6fb73e9f7de9dda631bcdbca564aa5435
TEMPLATE "{{- if .Messages }}
{{- range $index, $_ := .Messages }}
{{- if eq .Role "user" }}
{{- if and (eq (len (slice $.Messages $index)) 1) $.Tools }}[AVAILABLE_TOOLS] {{ $.Tools }}[/AVAILABLE_TOOLS]
{{- end }}[INST] {{ if and $.System (eq (len (slice $.Messages $index)) 1) }}{{ $.System }}

{{ end }}{{ .Content }}[/INST]
{{- else if eq .Role "assistant" }}
{{- if .Content }} {{ .Content }}
{{- else if .ToolCalls }}[TOOL_CALLS] [
{{- range .ToolCalls }}{"name": "{{ .Function.Name }}", "arguments": {{ .Function.Arguments }}}
{{- end }}]
{{- end }}</s>
{{- else if eq .Role "tool" }}[TOOL_RESULTS] {"content": {{ .Content }}} [/TOOL_RESULTS]
{{- end }}
{{- end }}
{{- else }}[INST] {{ if .System }}{{ .System }}

{{ end }}{{ .Prompt }}[/INST]
{{